In [0]:
import time
import hashlib
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%run ../00_utils/common

In [0]:
%run ../00_utils/quality

In [0]:
BATCH = batch_id()

## mstr_proveedores

In [0]:
def process_proveedores():
      print(" === Inicio process_proveedores ===")
      t0 = time.time()
      df = spark.table(f"{CATALOG}.bronze.mstr_proveedores")

      df = (df
            # Deduplicar por PK conservando el más reciente
            .withColumn("_rn", F.row_number().over(
                  Window.partitionBy("id_proveedor").orderBy(F.col("_ingested_at").desc())))
            .filter(F.col("_rn") == 1).drop("_rn")
            # Eliminar nulos en campo obligatorio
            .filter(F.col("id_proveedor").isNotNull())
            # Aplicar calificacion_calidad nula con mediana
            .withColumn("calificacion_calidad",
                  F.when(F.col("calificacion_calidad").isNull(),
                        F.percentile_approx("calificacion_calidad", 0.5).over(Window.partitionBy()))
                  .otherwise(F.col("calificacion_calidad")))
            .withColumn("activo", F.col("activo").cast("boolean"))
            .withColumn("_batch_id", F.lit(BATCH)))

      quality_report(df, "silver.mstr_proveedores", ["id_proveedor"], ["calificacion_calidad"], [])
      df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.mstr_proveedores")
      log_run("silver", "mstr_proveedores", df.count(), 0, time.time() - t0, batch=BATCH)
      print(f"  [DONE] silver.mstr_proveedores")

## mstr_articulos

In [0]:
def process_articulos():
    print(" === Inicio process_articulos ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.bronze.mstr_articulos")

    df = (df
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("art_id").orderBy(F.col("_ingested_at").desc())))
          .filter(F.col("_rn") == 1).drop("_rn")
          .filter(F.col("art_id").isNotNull() & F.col("id_proveedor").isNotNull())
          # Validar precio positivo — rechazar negativos
          .filter(F.col("precio_lista") > 0)
          .withColumn("precio_lista", F.col("precio_lista").cast("double"))
          .withColumn("peso_kg",
                      F.when(F.col("peso_kg").isNull(), F.lit(0.0))
                      .otherwise(F.col("peso_kg")))
          .withColumn("activo", F.col("activo").cast("boolean"))
          .withColumn("fec_alta", F.to_date("fec_alta"))
          .withColumn("_batch_id", F.lit(BATCH)))

    # Integridad referencial: art → proveedor
    df_provs = spark.table(f"{CATALOG}.silver.mstr_proveedores").select("id_proveedor")
    df_orphan = check_referential_integrity(df,df_provs, "id_proveedor", "id_proveedor", "mstr_articulos")
    if df_orphan.count():
        log_referential_errors(df_orphan, "mstr_articulos", "id_proveedor sin match", BATCH)
    df = df.join(df_provs, on="id_proveedor", how="inner")

    quality_report(df, "silver.mstr_articulos", ["art_id"], ["peso_kg"], ["precio_lista"])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.mstr_articulos")
    log_run("silver", "mstr_articulos", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] silver.mstr_articulos")

## mstr_tiendas

In [0]:
def process_tiendas():
    print(" === Inicio process_tiendas ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.bronze.mstr_tiendas")

    tipo_map = {
        "Hipermercado": "HIPERMERCADO",
        "Supermercado Barrio": "SUPERMERCADO",
        "Tienda Conveniencia": "CONVENIENCIA",
    }
    mapping_expr = F.create_map([F.lit(k) for pair in tipo_map.items() for k in pair])

    df = (df
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("id_tienda").orderBy(F.col("_ingested_at").desc())))
          .filter(F.col("_rn") == 1).drop("_rn")
          .filter(F.col("id_tienda").isNotNull())
          .withColumn("tipo_tienda", mapping_expr[F.col("tipo_tienda")])
          .withColumn("metros_cuadrados", F.col("metros_cuadrados").cast("integer"))
          .withColumn("activo", F.col("activo").cast("boolean"))
          .withColumn("fec_apertura", F.to_date("fec_apertura"))
          .withColumn("_batch_id", F.lit(BATCH)))

    quality_report(df, "silver.mstr_tiendas", ["id_tienda"], [], ["metros_cuadrados"])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.mstr_tiendas")
    log_run("silver", "mstr_tiendas", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] silver.mstr_tiendas")

## crm_miembros

In [0]:
def process_miembros():
    print(" === Inicio process_miembros ===")
    t0 = time.time()
    df = spark.table(f"{CATALOG}.bronze.crm_miembros")

    # Moda de rango_edad por canal para manejar nulos
    canal_moda = (df.filter(F.col("rango_edad").isNotNull())
                  .groupBy("canal_pref", "rango_edad").count()
                  .withColumn("_rn", F.row_number().over(Window.partitionBy("canal_pref").orderBy(F.col("count").desc())))
                  .filter(F.col("_rn") == 1)
                  .drop("count", "_rn")
                  .withColumnRenamed("rango_edad", "moda_edad"))

    df = (df
          .withColumn("_rn", F.row_number().over(
              Window.partitionBy("id_miembro").orderBy(F.col("_ingested_at").desc())))
          .filter(F.col("_rn") == 1).drop("_rn")
          .filter(F.col("id_miembro").isNotNull())
          # Estandarizar género
          .withColumn("genero",
              F.when(F.col("genero").isin("M", "F"), F.col("genero"))
               .otherwise(F.lit("No informado")))
          .withColumn("activo", F.col("activo").cast("boolean"))
          .withColumn("fec_registro", F.to_date("fec_registro"))
          .withColumn("fec_ultima_compra", F.to_date("fec_ultima_compra"))
          # PII: hash del id_miembro
          .withColumn("id_miembro_hash", F.sha2(F.col("id_miembro"), 256))
          .withColumn("_batch_id", F.lit(BATCH)))

    # Aplicar rango_edad nulo con moda del canal preferido
    df = (df.join(canal_moda, on="canal_pref", how="left")
            .withColumn("rango_edad",
                F.when(F.col("rango_edad").isNull(), F.col("moda_edad"))
                 .otherwise(F.col("rango_edad")))
            .drop("moda_edad"))

    quality_report(df, "silver.crm_miembros", ["id_miembro"], ["genero", "fec_ultima_compra"], [])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.crm_miembros")
    log_run("silver", "crm_miembros", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] silver.crm_miembros")

## Execution

In [0]:
print(f"Silver Dimensions | batch={BATCH}")
try: process_proveedores()
except Exception as e: print(f"  [ERROR] mstr_proveedores: {e}")
try: process_articulos()
except Exception as e: print(f"  [ERROR] mstr_articulos: {e}")
try: process_tiendas()
except Exception as e: print(f"  [ERROR] mstr_tiendas: {e}")
try: process_miembros()
except Exception as e: print(f"  [ERROR] crm_miembros: {e}")
print("\n[DONE] Silver dimensions completado.")